# Trabajo Práctico - Diseño de Solución de Datos
## Sistema de Agricultura de Precisión basado en IoT y LoRaWAN

---

# 1. Análisis del caso de uso

## 1.1 Descripción del caso de uso
El proyecto consiste en el diseño de una solución de datos para un sistema de Agricultura de Precisión basado en dispositivos IoT y comunicaciones LoRaWAN. El sistema permite monitorear establecimientos agrícolas mediante dispositivos instalados en el campo, almacenar las mediciones generadas y brindar información para el monitoreo, la gestión del riego y futuras aplicaciones de Inteligencia Artificial.

## 1.2 Problema que busca resolver
Los dispositivos instalados en el campo generan información de forma continua y distribuida. La solución propuesta busca centralizar estos datos, administrar la infraestructura agrícola y conservar el historial de mediciones para facilitar el monitoreo, el análisis y la toma de decisiones.


## 1.3 Usuarios principales
El sistema contempla tres perfiles de usuario:
- **Operador:** consulta dispositivos, mediciones, gráficos y alarmas.
- **Configurador:** además de las funciones del Operador, administra configuraciones y alarmas.
- **Administrador:** administra usuarios, perfiles y permisos, además de todas las funciones anteriores.


## 1.4 Procesos y funcionalidades
El sistema deberá permitir:
- Administrar establecimientos, lotes, sectores y pivotes.
- Registrar y administrar dispositivos.
- Gestionar el historial de instalación de los dispositivos.
- Almacenar mediciones y datos de comunicación.
- Monitorear variables en tiempo real.
- Consultar información histórica.
- Administrar alarmas.
- Gestionar usuarios y permisos.


## 1.5 Información que gestiona el sistema
La solución administra información correspondiente a:
- Establecimientos agrícolas.
- Lotes, sectores y pivotes.
- Dispositivos IoT.
- Historial de instalaciones.
- Mediciones.
- Gateways y datos de comunicación.
- Usuarios y perfiles.
- Configuraciones y alarmas.

## 1.6 Riesgos relacionados con los datos
Los principales riesgos considerados son:
- Pérdida de mediciones.
- Inconsistencias en la ubicación de los dispositivos.
- Accesos no autorizados.
- Modificaciones indebidas de configuraciones.
- Crecimiento del volumen de datos.
- Pérdida de integridad de la información.


## 1.7 Principales decisiones de diseño
El diseño de la solución priorizará la integridad, trazabilidad y escalabilidad de los datos. Para ello, se considerará inicialmente utilizar una base de datos relacional PostgreSQL como plataforma principal, complementada con la extensión TimescaleDB para el almacenamiento eficiente de series temporales. Además, las variables medidas por los dispositivos se almacenarán en formato JSONB, permitiendo representar distintos tipos de mediciones de manera flexible.

---

# 2. Relevamiento de datos necesarios
La solución a desarrollar deberá almacenar o consultar los siguientes datos, clasificados según las categorías propuestas:

## 2.1 Datos estructurados
Se identifican los siguientes:
- Campos.
- Lotes.
- Sectores.
- Pivotes.
- Categorías de dispositivos.
- Tipos de dispositivos.
- Dispositivos.
- Instalaciones.
- Gateways.
- Usuarios.
- Perfiles.
- Alarmas.
- Configuraciones.

## 2.2 Datos semiestructurados
Las “mediciones”, ya que las variables medidas difieren según el tipo de dispositivo. Por ejemplo:
1. Estaciones meteorológicas:
    - temperatura.
    - humedad relativa.
    - velocidad y dirección del viento.
    - radiación solar.
    - precipitaciones.
2. sondas de suelo: por cada nivel (6 niveles):
    - humedad.
    - temperatura.
    - conductividad eléctrica.

Las variables medidas se almacenarán en formato JSONB, permitiendo registrar distintos conjuntos de valores sin modificar la estructura de la tabla de mediciones.


## 2.3 Datos no estructurados
No están contemplados por ahora.

## 2.4 Datos operacionales
Consideramos como datos operacionales a aquellos utilizados por la aplicación durante su funcionamiento diario, necesarios para realizar las tareas de monitoreo y administración.
Entre ellos se encuentran:
- Ubicación de sensores.
- Mediciones.
- Alarmas.
- Configuraciones.

## 2.5 Datos analíticos
En esta categoría básicamente se encuentra el histórico de mediciones, utilizado para realizar análisis y apoyar la toma de decisiones.


## 2.6 Datos sensibles
El sistema administra información que requiere protección, entre ella:
- Credenciales de acceso.
- Datos personales de los usuarios.
- Configuraciones del sistema.

## 2.7 Datos de auditoría y trazabilidad
El sistema conservará información que permita reconstruir eventos y realizar auditorías, incluyendo:
- Historial de instalación de los dispositivos.
- Historial de asignación de pivotes a lotes.

(Estos datos permitirán relacionar, por ejemplo, la influencia del riego en las variables medidas en suelo).

- Registros de recepción de mediciones.
- Fecha y hora de cada medición.

(Para corroborar que no se hayan perdido mensajes).
- Cambios realizados sobre configuraciones y alarmas.


## 2.8 Ejemplos de datos

---

# 8. Implementación mínima realizada

## 8.1 Esquema físico
El esquema físico se implementó en PostgreSQL 16 con la extensión TimescaleDB, en `db/estructura/01_create_tables.sql`. El script define 15 tablas, organizadas en cinco bloques:

1. **Organización del establecimiento**: `campo`, `lote`, `sector`, `pivote`, `asignacion_pivote`.
2. **Dispositivos IoT**: `categoria_dispositivo`, `tipo_dispositivo`, `variable`, `tipo_variable`, `dispositivo`, `instalacion_dispositivo`.
3. **Mediciones**: `gateway`, `medicion`.
4. **Alarmas**: `regla_alarma`, `evento_alarma`, `alarma_dispositivo`.
5. **Usuarios**: `perfil`, `usuario`.

El script se ejecuta sobre una base recién creada; no usa `IF NOT EXISTS` porque se apoya en el mecanismo de inicialización de la imagen de Postgres (ver 8.3), que solo corre una vez, sobre un volumen vacío.

## 8.2 Decisiones de diseño reflejadas en el DDL

**Hipertabla TimescaleDB para `medicion`.** La tabla de mediciones se convierte en hipertabla mediante `create_hypertable('medicion', 'fecha_hora')`, particionando internamente por tiempo. Es la tabla con mayor volumen esperado (una fila por dispositivo y transmisión), por lo que necesita el patrón de escritura y purga que ofrece TimescaleDB en lugar de una tabla relacional simple.

**JSONB para `valores_medidos`.** Cada tipo de dispositivo mide un conjunto distinto de variables (una estación meteorológica no mide lo mismo que una sonda de suelo). En vez de modelar una columna por variable o una tabla EAV, `medicion.valores_medidos` es JSONB, y se indexa con GIN (`ix_medicion_valores_gin`) para soportar filtros por contenido (operador `@>`) sin escanear toda la tabla.

**Instalación polimórfica de dispositivos.** Un dispositivo se instala en un campo, un sector o un pivote, nunca en más de uno a la vez. Se modeló con tres columnas FK nullable (`id_campo`, `id_sector`, `id_pivote`) en `instalacion_dispositivo` más un `CHECK` que exige que exactamente una esté presente:

```sql
CHECK (
    (id_campo IS NOT NULL)::INTEGER
    + (id_sector IS NOT NULL)::INTEGER
    + (id_pivote IS NOT NULL)::INTEGER = 1
)
```

Se prefirió esto a una tabla `ubicacion` genérica con `tipo` + `id_referencia` porque mantiene las FK reales de PostgreSQL (integridad referencial verificada por el motor), a costa de tener tres columnas nullable en vez de dos.

**Historial temporal.** `instalacion_dispositivo` y `asignacion_pivote` registran `fecha_inicio`/`fecha_fin`, donde `fecha_fin IS NULL` marca el registro activo. Un índice único parcial (`WHERE fecha_fin IS NULL`) garantiza que un mismo dispositivo o pivote no tenga más de una fila activa a la vez, sin depender de que la aplicación lo respete:

```sql
CREATE UNIQUE INDEX ux_instalacion_dispositivo_activa
    ON instalacion_dispositivo(id_dispositivo)
    WHERE fecha_fin IS NULL;
```

**Restricciones de integridad adicionales.** `regla_alarma` exige al menos un umbral definido (`umbral_inferior IS NOT NULL OR umbral_superior IS NOT NULL`); `dispositivo` restringe sus estados a valores válidos (`estado_operativo IN ('on', 'off')`, etc.) mediante `CHECK`; `usuario.email` es `UNIQUE`; las relaciones N:M (`tipo_variable`, `alarma_dispositivo`) se resuelven con tablas intermedias de clave primaria compuesta.

## 8.3 Entorno de ejecución

El entorno se levanta con `docker-compose.yml`, usando la imagen `timescale/timescaledb:latest-pg16`. El script `01_create_tables.sql` se monta en `/docker-entrypoint-initdb.d/`, que Postgres ejecuta automáticamente la primera vez que arranca sobre un volumen vacío — no hace falta correr el DDL a mano. Las credenciales y el nombre de la base se parametrizan por variables de entorno (`.env`, a partir de `.env.example`) para no versionar contraseñas.

Con esto, levantar el esquema completo desde cero se reduce a `docker compose up -d`.

---

# 9. Datos de ejemplo utilizados

## 9.1 Generación de datos sintéticos

Los datos de ejemplo se generan con un script en Python (`db/datos/generar_datos.py` + `db/datos/main.py`), usando `psycopg2` para insertar contra la base y `Faker` (locale `es_AR`) para los datos con apariencia realista (nombres, emails, ubicaciones). El script puebla las 15 tablas del esquema, respetando el orden de dependencias entre ellas y las restricciones definidas en el DDL.

## 9.2 Volumen generado

| Entidad | Cantidad |
| --- | --- |
| Campos | 3 |
| Lotes | 9 (3 por campo) |
| Sectores | 18 (2 por lote) |
| Pivotes | 6 (2 por campo) |
| Asignaciones de pivote | 6 |
| Categorías y tipos de dispositivo | según catálogo fijo (suelo, riego, meteorológico) |
| Dispositivos | 3 por tipo |
| Instalaciones de dispositivo | 1 por dispositivo (activa) |
| Gateways | 2 |
| Mediciones | 800 |
| Reglas de alarma | 4 |
| Eventos de alarma | 3 por regla (12 en total) |
| Usuarios | 6 |

Son volúmenes pequeños a propósito: alcanzan para validar entidades, relaciones y restricciones (la consigna no exige un dataset real ni de gran escala), sin complicar la verificación manual de los resultados.

## 9.3 Coherencia de los datos generados

Los datos no son aleatorios sin criterio: respetan la semántica del dominio.

- **`valores_medidos` varía según el tipo de dispositivo.** Una sonda de suelo genera JSON con humedad/temperatura/conductividad por nivel; una estación meteorológica genera temperatura, humedad relativa, viento, radiación y precipitación, coherente con lo relevado en la sección 2.2.
- **Las reglas de alarma tienen umbrales con sentido físico.** Por ejemplo, la regla de batería baja dispara con `umbral_inferior`, no `umbral_superior` (un valor de batería por debajo de un límite es el caso anómalo). Esto se verificó revisando los `valor_detectado` generados en `evento_alarma` contra el umbral de cada regla.
- **El historial temporal es consistente.** Cada dispositivo y cada pivote tiene exactamente una instalación/asignación activa (`fecha_fin IS NULL`) al finalizar la carga, reforzado por los índices únicos parciales del DDL.

## 9.4 Exportación de ejemplo

El script exporta una muestra de 100 mediciones (join contra `dispositivo` y `tipo_dispositivo`) a `data/ejemplos/mediciones.csv`, serializando `valores_medidos` como JSON válido (`json.dumps`) en lugar de la representación de string por defecto de Python, para que el CSV sea reutilizable por otras herramientas.